In [2]:
import sqlite3
import random
import uuid
import os
from datetime import datetime, timedelta
from faker import Faker

DB_PATH = "techstyle.db"

fake = Faker("pt_BR")

# =========================================================
# CONFIGURAÇÕES DE VOLUME - EMPRESA DE MÉDIO PORTE
# =========================================================
N_CUSTOMERS = 18000
ADDR_MIN = 1
ADDR_MAX = 3

CATEGORY_NAMES = [
    "Camisetas", "Calças", "Vestidos", "Jaquetas",
    "Tênis", "Sapatos", "Sandálias", "Bolsas",
    "Relógios", "Óculos", "Acessórios", "Moda Íntima"
]

BRAND_NAMES = [
    "Nike", "Adidas", "Puma", "Zara", "Hering", "Reserva", "Arezzo", "Farm",
    "Colcci", "Levis", "Fila", "Vans", "Converse", "Mizuno", "Lacoste",
    "Tommy Hilfiger", "Calvin Klein", "Diesel", "Guess", "Ellus"
]
N_BRANDS_EXTRA = 65

N_PRODUCTS = 4200
N_WAREHOUSES = 6
N_COUPONS = 120
N_CAMPAIGNS = 36
N_ORDERS = 52000

ORDER_ITEMS_MIN = 1
ORDER_ITEMS_MAX = 4

RETURN_RATE = 0.065
REVIEW_RATE = 0.18

# =========================================================
# DDL
# =========================================================
DDL = """
CREATE TABLE IF NOT EXISTS customer_segments (
    segment_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    segment_name TEXT NOT NULL UNIQUE,
    description  TEXT,
    created_at   TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS customers (
    customer_id  INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name    TEXT NOT NULL,
    email        TEXT NOT NULL UNIQUE,
    phone        TEXT,
    gender       TEXT,
    birth_date   TEXT,
    signup_date  TEXT NOT NULL,
    status       TEXT NOT NULL DEFAULT 'active',
    created_at   TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at   TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS customer_segments_log (
    log_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    segment_id  INTEGER NOT NULL REFERENCES customer_segments(segment_id),
    start_date  TEXT    NOT NULL,
    end_date    TEXT,
    is_current  INTEGER NOT NULL DEFAULT 1,
    created_at  TEXT    NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS addresses (
    address_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id   INTEGER NOT NULL REFERENCES customers(customer_id),
    address_type  TEXT NOT NULL,
    street        TEXT NOT NULL,
    street_number TEXT NOT NULL,
    complement    TEXT,
    neighborhood  TEXT,
    city          TEXT NOT NULL,
    state         TEXT NOT NULL,
    zip_code      TEXT NOT NULL,
    country       TEXT NOT NULL DEFAULT 'Brasil',
    is_default    INTEGER NOT NULL DEFAULT 0,
    created_at    TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at    TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS categories (
    category_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    parent_category_id INTEGER REFERENCES categories(category_id),
    category_name      TEXT NOT NULL UNIQUE,
    description        TEXT,
    is_active          INTEGER NOT NULL DEFAULT 1,
    created_at         TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at         TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS brands (
    brand_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    brand_name     TEXT NOT NULL UNIQUE,
    origin_country TEXT,
    is_active      INTEGER NOT NULL DEFAULT 1,
    created_at     TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at     TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS products (
    product_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    category_id  INTEGER NOT NULL REFERENCES categories(category_id),
    brand_id     INTEGER NOT NULL REFERENCES brands(brand_id),
    product_name TEXT NOT NULL,
    sku          TEXT NOT NULL UNIQUE,
    color        TEXT,
    size         TEXT,
    unit_cost    REAL NOT NULL,
    unit_price   REAL NOT NULL,
    launch_date  TEXT,
    status       TEXT NOT NULL DEFAULT 'active',
    created_at   TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at   TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS warehouses (
    warehouse_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    warehouse_name TEXT NOT NULL,
    city           TEXT NOT NULL,
    state          TEXT NOT NULL,
    country        TEXT NOT NULL DEFAULT 'Brasil',
    created_at     TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at     TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS inventory (
    inventory_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id         INTEGER NOT NULL REFERENCES products(product_id),
    warehouse_id       INTEGER NOT NULL REFERENCES warehouses(warehouse_id),
    quantity_available INTEGER NOT NULL DEFAULT 0,
    reorder_level      INTEGER NOT NULL DEFAULT 0,
    last_restock_date  TEXT,
    created_at         TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at         TEXT NOT NULL DEFAULT (datetime('now')),
    UNIQUE (product_id, warehouse_id)
);

CREATE TABLE IF NOT EXISTS coupons (
    coupon_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    coupon_code     TEXT NOT NULL UNIQUE,
    description     TEXT,
    discount_type   TEXT NOT NULL,
    discount_value  REAL NOT NULL,
    start_date      TEXT NOT NULL,
    end_date        TEXT NOT NULL,
    min_order_value REAL,
    is_active       INTEGER NOT NULL DEFAULT 1,
    created_at      TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at      TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS campaigns (
    campaign_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    campaign_name TEXT NOT NULL,
    channel       TEXT NOT NULL,
    start_date    TEXT NOT NULL,
    end_date      TEXT NOT NULL,
    budget        REAL,
    objective     TEXT,
    created_at    TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at    TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS campaign_coupons (
    campaign_id INTEGER NOT NULL REFERENCES campaigns(campaign_id),
    coupon_id   INTEGER NOT NULL REFERENCES coupons(coupon_id),
    created_at  TEXT NOT NULL DEFAULT (datetime('now')),
    PRIMARY KEY (campaign_id, coupon_id)
);

CREATE TABLE IF NOT EXISTS orders (
    order_id            INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id         INTEGER NOT NULL REFERENCES customers(customer_id),
    shipping_address_id INTEGER NOT NULL REFERENCES addresses(address_id),
    coupon_id           INTEGER REFERENCES coupons(coupon_id),
    campaign_id         INTEGER REFERENCES campaigns(campaign_id),
    order_date          TEXT    NOT NULL,
    status              TEXT    NOT NULL,
    subtotal_amount     REAL    NOT NULL,
    discount_amount     REAL    NOT NULL DEFAULT 0,
    shipping_amount     REAL    NOT NULL DEFAULT 0,
    total_amount        REAL    NOT NULL,
    created_at          TEXT    NOT NULL DEFAULT (datetime('now')),
    updated_at          TEXT    NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS order_items (
    order_item_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id        INTEGER NOT NULL REFERENCES orders(order_id),
    product_id      INTEGER NOT NULL REFERENCES products(product_id),
    quantity        INTEGER NOT NULL,
    unit_price      REAL    NOT NULL,
    unit_cost       REAL    NOT NULL,
    discount_amount REAL    NOT NULL DEFAULT 0,
    line_total      REAL    NOT NULL,
    created_at      TEXT    NOT NULL DEFAULT (datetime('now')),
    updated_at      TEXT    NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS payments (
    payment_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id         INTEGER NOT NULL REFERENCES orders(order_id),
    payment_date     TEXT,
    payment_method   TEXT NOT NULL,
    status           TEXT NOT NULL,
    installments     INTEGER NOT NULL DEFAULT 1,
    paid_amount      REAL    NOT NULL,
    transaction_code TEXT,
    created_at       TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at       TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS shipments (
    shipment_id             INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id                INTEGER NOT NULL UNIQUE REFERENCES orders(order_id),
    carrier_name            TEXT,
    tracking_code           TEXT,
    shipment_date           TEXT,
    estimated_delivery_date TEXT,
    actual_delivery_date    TEXT,
    status                  TEXT NOT NULL,
    freight_cost            REAL NOT NULL DEFAULT 0,
    created_at              TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at              TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS returns (
    return_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    order_item_id INTEGER NOT NULL REFERENCES order_items(order_item_id),
    return_date   TEXT    NOT NULL,
    return_reason TEXT    NOT NULL,
    refund_amount REAL    NOT NULL,
    status        TEXT    NOT NULL,
    created_at    TEXT    NOT NULL DEFAULT (datetime('now')),
    updated_at    TEXT    NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS product_reviews (
    review_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id  INTEGER NOT NULL REFERENCES products(product_id),
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    order_id    INTEGER REFERENCES orders(order_id),
    rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    review_text TEXT,
    review_date TEXT NOT NULL,
    created_at  TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at  TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE INDEX IF NOT EXISTS idx_orders_date         ON orders (order_date DESC);
CREATE INDEX IF NOT EXISTS idx_orders_customer     ON orders (customer_id, order_date DESC);
CREATE INDEX IF NOT EXISTS idx_orders_status       ON orders (status);
CREATE INDEX IF NOT EXISTS idx_orders_campaign     ON orders (campaign_id);
CREATE INDEX IF NOT EXISTS idx_order_items_order   ON order_items (order_id);
CREATE INDEX IF NOT EXISTS idx_order_items_product ON order_items (product_id);
CREATE INDEX IF NOT EXISTS idx_segments_current    ON customer_segments_log (customer_id, is_current);
"""

# =========================================================
# FUNÇÕES AUXILIARES
# =========================================================
def random_date(start_days=365):
    return datetime.now() - timedelta(days=random.randint(0, start_days))

def maybe(value, probability=0.5):
    return value if random.random() < probability else None

def build_review_text(rating):
    review_texts_pos = [
        "Produto excelente, superou minhas expectativas.",
        "Ótimo acabamento e entrega rápida.",
        "Muito confortável e de boa qualidade.",
        "Gostei bastante, compraria novamente.",
        "Bom custo-benefício."
    ]

    review_texts_neu = [
        "Produto ok, atende ao esperado.",
        "Nada fora do normal, mas cumpre o que promete.",
        "Qualidade razoável pelo preço.",
        "Entrega dentro do prazo e produto correto."
    ]

    review_texts_neg = [
        "Esperava mais pela faixa de preço.",
        "O produto veio diferente da foto.",
        "Não gostei muito do material.",
        "Acabamento deixou a desejar."
    ]

    if rating >= 4:
        return random.choice(review_texts_pos)
    if rating == 3:
        return random.choice(review_texts_neu)
    return random.choice(review_texts_neg)

# =========================================================
# RECRIA BANCO
# =========================================================
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()
cur.executescript(DDL)

# =========================================================
# 1. CUSTOMER SEGMENTS
# =========================================================
segments = [
    ("VIP", "Clientes com alta recorrência e ticket médio elevado"),
    ("Novo Cliente", "Clientes recém-cadastrados"),
    ("Inativo", "Clientes sem compras recentes"),
    ("Promocional", "Clientes sensíveis a desconto"),
    ("Frequente", "Clientes com compras recorrentes"),
    ("Alto Potencial", "Clientes com boa propensão de recompra")
]

for segment_name, description in segments:
    cur.execute("""
        INSERT INTO customer_segments (segment_name, description)
        VALUES (?, ?)
    """, (segment_name, description))

# =========================================================
# 2. CUSTOMERS
# =========================================================
for i in range(N_CUSTOMERS):
    cur.execute("""
        INSERT INTO customers (
            full_name, email, phone, gender, birth_date, signup_date, status
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        fake.name(),
        f"cliente{i+1}_{uuid.uuid4().hex[:6]}@email.com",
        fake.phone_number(),
        random.choice(["M", "F", "Outro", None]),
        fake.date_of_birth(minimum_age=18, maximum_age=75).strftime("%Y-%m-%d"),
        random_date(1825).strftime("%Y-%m-%d"),
        random.choices(["active", "inactive", "blocked"], weights=[92, 7, 1], k=1)[0]
    ))

conn.commit()

# =========================================================
# 3. CUSTOMER SEGMENT LOG
# =========================================================
for customer_id in range(1, N_CUSTOMERS + 1):
    segment_id = random.randint(1, len(segments))
    cur.execute("""
        INSERT INTO customer_segments_log (
            customer_id, segment_id, start_date, is_current
        )
        VALUES (?, ?, ?, 1)
    """, (
        customer_id,
        segment_id,
        random_date(730).strftime("%Y-%m-%d")
    ))

conn.commit()

# =========================================================
# 4. ADDRESSES
# =========================================================
bairro_fn = getattr(fake, "bairro", None)

for customer_id in range(1, N_CUSTOMERS + 1):
    n_addr = random.randint(ADDR_MIN, ADDR_MAX)
    default_idx = random.randint(1, n_addr)

    for i in range(1, n_addr + 1):
        neighborhood = bairro_fn() if callable(bairro_fn) else fake.city_suffix()

        complement = random.choice([
            f"Apto {random.randint(1, 999)}",
            f"Bloco {random.choice(['A', 'B', 'C', 'D'])}",
            f"Casa {random.randint(1, 50)}",
            "Fundos",
            "Sala 2"
        ]) if random.random() < 0.25 else None

        cur.execute("""
            INSERT INTO addresses (
                customer_id, address_type, street, street_number, complement,
                neighborhood, city, state, zip_code, country, is_default
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            customer_id,
            random.choice(["residencial", "comercial"]),
            fake.street_name(),
            str(random.randint(1, 9999)),
            complement,
            neighborhood,
            fake.city(),
            fake.state_abbr(),
            fake.postcode(),
            "Brasil",
            1 if i == default_idx else 0
        ))

# =========================================================
# 5. CATEGORIES
# =========================================================
for category_name in CATEGORY_NAMES:
    cur.execute("""
        INSERT INTO categories (category_name, description, is_active)
        VALUES (?, ?, 1)
    """, (category_name, f"Categoria {category_name}"))

conn.commit()

# =========================================================
# 6. BRANDS
# =========================================================
for brand_name in BRAND_NAMES:
    cur.execute("""
        INSERT INTO brands (brand_name, origin_country, is_active)
        VALUES (?, ?, 1)
    """, (brand_name, fake.country()))

for _ in range(N_BRANDS_EXTRA):
    cur.execute("""
        INSERT INTO brands (brand_name, origin_country, is_active)
        VALUES (?, ?, 1)
    """, (fake.unique.company(), fake.country()))

conn.commit()

# =========================================================
# 7. PRODUCTS
# =========================================================
colors = ["Preto", "Branco", "Azul", "Cinza", "Vermelho", "Verde", "Bege", "Rosa"]
sizes = ["PP", "P", "M", "G", "GG", "36", "38", "40", "42", "44", "Único"]
total_brands = len(BRAND_NAMES) + N_BRANDS_EXTRA

for _ in range(N_PRODUCTS):
    cost = round(random.uniform(18, 220), 2)
    markup = random.uniform(1.8, 3.4)
    price = round(cost * markup, 2)

    cur.execute("""
        INSERT INTO products (
            category_id, brand_id, product_name, sku, color, size,
            unit_cost, unit_price, launch_date, status
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        random.randint(1, len(CATEGORY_NAMES)),
        random.randint(1, total_brands),
        f"{fake.word().capitalize()} {fake.word().capitalize()}",
        f"SKU-{uuid.uuid4().hex[:12].upper()}",
        random.choice(colors),
        random.choice(sizes),
        cost,
        price,
        random_date(1460).strftime("%Y-%m-%d"),
        random.choices(["active", "inactive"], weights=[95, 5], k=1)[0]
    ))

conn.commit()

# =========================================================
# 8. WAREHOUSES
# =========================================================
warehouse_cities = [
    ("CD São Paulo", "São Paulo", "SP"),
    ("CD Rio de Janeiro", "Rio de Janeiro", "RJ"),
    ("CD Belo Horizonte", "Belo Horizonte", "MG"),
    ("CD Curitiba", "Curitiba", "PR"),
    ("CD Recife", "Recife", "PE"),
    ("CD Goiânia", "Goiânia", "GO"),
]

for name, city, state in warehouse_cities[:N_WAREHOUSES]:
    cur.execute("""
        INSERT INTO warehouses (warehouse_name, city, state, country)
        VALUES (?, ?, ?, ?)
    """, (name, city, state, "Brasil"))

conn.commit()

# =========================================================
# 9. INVENTORY
# =========================================================
for product_id in range(1, N_PRODUCTS + 1):
    for warehouse_id in range(1, N_WAREHOUSES + 1):
        qty = random.randint(0, 350)
        reorder = random.randint(15, 80)

        cur.execute("""
            INSERT INTO inventory (
                product_id, warehouse_id, quantity_available,
                reorder_level, last_restock_date
            )
            VALUES (?, ?, ?, ?, ?)
        """, (
            product_id,
            warehouse_id,
            qty,
            reorder,
            random_date(180).strftime("%Y-%m-%d")
        ))

conn.commit()

# =========================================================
# 10. COUPONS
# =========================================================
for _ in range(N_COUPONS):
    start_dt = random_date(365)
    end_dt = start_dt + timedelta(days=random.randint(15, 120))

    cur.execute("""
        INSERT INTO coupons (
            coupon_code, description, discount_type, discount_value,
            start_date, end_date, min_order_value, is_active
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        f"PROMO-{uuid.uuid4().hex[:8].upper()}",
        fake.sentence(nb_words=4),
        random.choice(["percent", "fixed"]),
        round(random.uniform(10, 80), 2),
        start_dt.strftime("%Y-%m-%d"),
        end_dt.strftime("%Y-%m-%d"),
        round(random.uniform(100, 500), 2),
        random.choices([1, 0], weights=[75, 25], k=1)[0]
    ))

conn.commit()

# =========================================================
# 11. CAMPAIGNS
# =========================================================
for _ in range(N_CAMPAIGNS):
    start_dt = random_date(365)
    end_dt = start_dt + timedelta(days=random.randint(10, 90))

    cur.execute("""
        INSERT INTO campaigns (
            campaign_name, channel, start_date, end_date, budget, objective
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        fake.catch_phrase(),
        random.choice(["email", "ads", "social", "affiliate", "influencer"]),
        start_dt.strftime("%Y-%m-%d"),
        end_dt.strftime("%Y-%m-%d"),
        round(random.uniform(5000, 120000), 2),
        random.choice([
            "Aquisição de clientes",
            "Reativação da base",
            "Aumento do ticket médio",
            "Liquidação sazonal",
            "Lançamento de coleção"
        ])
    ))

conn.commit()

# =========================================================
# 12. CAMPAIGN_COUPONS
# =========================================================
for campaign_id in range(1, N_CAMPAIGNS + 1):
    linked_coupons = random.sample(
        range(1, N_COUPONS + 1),
        k=random.randint(1, min(5, N_COUPONS))
    )
    for coupon_id in linked_coupons:
        cur.execute("""
            INSERT OR IGNORE INTO campaign_coupons (campaign_id, coupon_id)
            VALUES (?, ?)
        """, (campaign_id, coupon_id))

conn.commit()

# =========================================================
# MAPA DE ENDEREÇOS POR CLIENTE
# =========================================================
cur.execute("SELECT customer_id, address_id FROM addresses")
address_rows = cur.fetchall()

addresses_by_customer = {}
for customer_id, address_id in address_rows:
    addresses_by_customer.setdefault(customer_id, []).append(address_id)

# =========================================================
# 13. ORDERS
# =========================================================
for _ in range(N_ORDERS):
    customer_id = random.randint(1, N_CUSTOMERS)
    customer_addresses = addresses_by_customer.get(customer_id)

    if not customer_addresses:
        continue

    shipping_address_id = random.choice(customer_addresses)

    subtotal = round(random.uniform(79, 1250), 2)
    discount = round(random.uniform(0, subtotal * 0.20), 2)
    shipping = round(random.uniform(0, 45), 2)
    total = round(subtotal - discount + shipping, 2)

    has_coupon = random.random() < 0.32
    has_campaign = random.random() < 0.28

    cur.execute("""
        INSERT INTO orders (
            customer_id, shipping_address_id, coupon_id, campaign_id,
            order_date, status, subtotal_amount, discount_amount,
            shipping_amount, total_amount
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        customer_id,
        shipping_address_id,
        random.randint(1, N_COUPONS) if has_coupon else None,
        random.randint(1, N_CAMPAIGNS) if has_campaign else None,
        random_date(730).strftime("%Y-%m-%d"),
        random.choices(
            ["pending", "paid", "shipped", "delivered", "cancelled"],
            weights=[6, 10, 14, 66, 4],
            k=1
        )[0],
        subtotal,
        discount,
        shipping,
        total
    ))

conn.commit()

# =========================================================
# 14. ORDER ITEMS
# =========================================================
for order_id in range(1, N_ORDERS + 1):
    num_items = random.randint(ORDER_ITEMS_MIN, ORDER_ITEMS_MAX)
    product_ids = random.sample(range(1, N_PRODUCTS + 1), num_items)

    for product_id in product_ids:
        cur.execute("""
            SELECT unit_price, unit_cost
            FROM products
            WHERE product_id = ?
        """, (product_id,))
        product = cur.fetchone()

        if not product:
            continue

        unit_price, unit_cost = product
        qty = random.randint(1, 3)
        item_discount = round(random.uniform(0, unit_price * qty * 0.10), 2)
        line_total = round(unit_price * qty - item_discount, 2)

        cur.execute("""
            INSERT INTO order_items (
                order_id, product_id, quantity, unit_price, unit_cost,
                discount_amount, line_total
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            order_id,
            product_id,
            qty,
            unit_price,
            unit_cost,
            item_discount,
            line_total
        ))

conn.commit()

# =========================================================
# 15. PAYMENTS
# =========================================================
for order_id in range(1, N_ORDERS + 1):
    cur.execute("""
        SELECT total_amount, status, order_date
        FROM orders
        WHERE order_id = ?
    """, (order_id,))
    row = cur.fetchone()

    if not row:
        continue

    total_amount, order_status, order_date = row

    if order_status == "cancelled":
        pay_status = "failed"
        paid_amount = 0
        payment_date = None
    elif order_status == "pending":
        pay_status = "pending"
        paid_amount = 0
        payment_date = None
    else:
        pay_status = "paid"
        paid_amount = total_amount
        base_dt = datetime.strptime(order_date, "%Y-%m-%d")
        payment_date = (base_dt + timedelta(days=random.randint(0, 2))).strftime("%Y-%m-%d")

    cur.execute("""
        INSERT INTO payments (
            order_id, payment_date, payment_method, status,
            installments, paid_amount, transaction_code
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        order_id,
        payment_date,
        random.choices(
            ["credit_card", "pix", "boleto", "debit_card"],
            weights=[48, 30, 12, 10],
            k=1
        )[0],
        pay_status,
        random.randint(1, 8),
        paid_amount,
        uuid.uuid4().hex[:20].upper()
    ))

conn.commit()

# =========================================================
# 16. SHIPMENTS
# =========================================================
for order_id in range(1, N_ORDERS + 1):
    cur.execute("""
        SELECT status, order_date
        FROM orders
        WHERE order_id = ?
    """, (order_id,))
    row = cur.fetchone()

    if not row:
        continue

    order_status, order_date = row

    if order_status not in ["shipped", "delivered"]:
        continue

    order_dt = datetime.strptime(order_date, "%Y-%m-%d")
    shipment_date = order_dt + timedelta(days=random.randint(1, 4))
    estimated_delivery = shipment_date + timedelta(days=random.randint(3, 10))

    if order_status == "delivered":
        actual_delivery = estimated_delivery + timedelta(days=random.randint(-1, 3))
        shipment_status = "delivered"
    else:
        actual_delivery = None
        shipment_status = random.choice(["shipped", "in_transit"])

    cur.execute("""
        INSERT INTO shipments (
            order_id, carrier_name, tracking_code, shipment_date,
            estimated_delivery_date, actual_delivery_date, status, freight_cost
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        order_id,
        random.choice(["Correios", "Jadlog", "Loggi", "Total Express", "Sequoia"]),
        f"TRK{uuid.uuid4().hex[:12].upper()}",
        shipment_date.strftime("%Y-%m-%d"),
        estimated_delivery.strftime("%Y-%m-%d"),
        actual_delivery.strftime("%Y-%m-%d") if actual_delivery else None,
        shipment_status,
        round(random.uniform(12, 42), 2)
    ))

conn.commit()

# =========================================================
# 17. RETURNS
# =========================================================
cur.execute("""
    SELECT oi.order_item_id, oi.line_total
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.status = 'delivered'
""")
delivered_items = cur.fetchall()

num_returns = int(len(delivered_items) * RETURN_RATE)
sampled_returns = random.sample(delivered_items, min(num_returns, len(delivered_items)))

for order_item_id, line_total in sampled_returns:
    refund_amount = round(line_total * random.uniform(0.6, 1.0), 2)

    cur.execute("""
        INSERT INTO returns (
            order_item_id, return_date, return_reason, refund_amount, status
        )
        VALUES (?, ?, ?, ?, ?)
    """, (
        order_item_id,
        random_date(365).strftime("%Y-%m-%d"),
        random.choice([
            "Produto com defeito",
            "Tamanho incompatível",
            "Arrependimento da compra",
            "Item incorreto",
            "Avaria no transporte"
        ]),
        refund_amount,
        random.choices(
            ["requested", "approved", "refunded", "rejected"],
            weights=[10, 20, 60, 10],
            k=1
        )[0]
    ))

conn.commit()

# =========================================================
# 18. PRODUCT REVIEWS
# =========================================================
cur.execute("""
    SELECT DISTINCT o.order_id, o.customer_id, oi.product_id, o.order_date
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.status = 'delivered'
""")
delivered_purchases = cur.fetchall()

num_reviews = int(len(delivered_purchases) * REVIEW_RATE)
sampled_reviews = random.sample(delivered_purchases, min(num_reviews, len(delivered_purchases)))

for order_id, customer_id, product_id, order_date in sampled_reviews:
    rating = random.choices([1, 2, 3, 4, 5], weights=[4, 6, 15, 35, 40], k=1)[0]
    order_dt = datetime.strptime(order_date, "%Y-%m-%d")
    review_dt = order_dt + timedelta(days=random.randint(5, 45))

    cur.execute("""
        INSERT INTO product_reviews (
            product_id, customer_id, order_id, rating, review_text, review_date
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        product_id,
        customer_id,
        order_id,
        rating,
        build_review_text(rating),
        review_dt.strftime("%Y-%m-%d")
    ))

conn.commit()

# =========================================================
# RESUMO FINAL
# =========================================================
tables = [
    "customer_segments",
    "customers",
    "customer_segments_log",
    "addresses",
    "categories",
    "brands",
    "products",
    "warehouses",
    "inventory",
    "coupons",
    "campaigns",
    "campaign_coupons",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "returns",
    "product_reviews"
]

print("\nResumo da carga:")
for table in tables:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    print(f"{table:25} {count}")

conn.close()
print("\nDados fictícios inseridos com sucesso em", DB_PATH)


Resumo da carga:
customer_segments         6
customers                 18000
customer_segments_log     18000
addresses                 36018
categories                12
brands                    85
products                  4200
warehouses                6
inventory                 25200
coupons                   120
campaigns                 36
campaign_coupons          108
orders                    52000
order_items               130377
payments                  52000
shipments                 41571
returns                   5599
product_reviews           15505

Dados fictícios inseridos com sucesso em techstyle.db


In [6]:
import os
import sqlite3
import pandas as pd

pasta_saida = os.path.join(os.getcwd(), "csvs")
os.makedirs(pasta_saida, exist_ok=True)

conn = sqlite3.connect("techstyle.db")

queries = {
    "Segmentos_Clientes": "SELECT * FROM customer_segments",
    "Clientes": "SELECT * FROM customers",
    "Historico_Segmentos": "SELECT * FROM customer_segments_log",
    "Enderecos": "SELECT * FROM addresses",
    "Categorias": "SELECT * FROM categories",
    "Marcas": "SELECT * FROM brands",
    "Produtos": "SELECT * FROM products",
    "Centros_Distribuicao": "SELECT * FROM warehouses",
    "Estoque": "SELECT * FROM inventory",
    "Cupons": "SELECT * FROM coupons",
    "Campanhas": "SELECT * FROM campaigns",
    "Campanhas_Cupons": "SELECT * FROM campaign_coupons",
    "Pedidos": "SELECT * FROM orders",
    "Itens_de_Pedido": "SELECT * FROM order_items",
    "Pagamentos": "SELECT * FROM payments",
    "Envios": "SELECT * FROM shipments",
    "Devolucoes": "SELECT * FROM returns",
    "Avaliacoes": "SELECT * FROM product_reviews",
}

for nome_arquivo, sql in queries.items():
    df = pd.read_sql(sql, conn)
    caminho = os.path.join(pasta_saida, f"{nome_arquivo}.csv")
    df.to_csv(caminho, index=False, encoding="utf-8-sig")
    print(f"{nome_arquivo:<25} {len(df):>8} linhas  →  {caminho}")

conn.close()
print("\nTodos os CSVs foram salvos com sucesso.")

Segmentos_Clientes               6 linhas  →  /Users/mubotti/Downloads/Portifólio/TechStyle/csvs/Segmentos_Clientes.csv
Clientes                     18000 linhas  →  /Users/mubotti/Downloads/Portifólio/TechStyle/csvs/Clientes.csv
Historico_Segmentos          18000 linhas  →  /Users/mubotti/Downloads/Portifólio/TechStyle/csvs/Historico_Segmentos.csv
Enderecos                    36018 linhas  →  /Users/mubotti/Downloads/Portifólio/TechStyle/csvs/Enderecos.csv
Categorias                      12 linhas  →  /Users/mubotti/Downloads/Portifólio/TechStyle/csvs/Categorias.csv
Marcas                          85 linhas  →  /Users/mubotti/Downloads/Portifólio/TechStyle/csvs/Marcas.csv
Produtos                      4200 linhas  →  /Users/mubotti/Downloads/Portifólio/TechStyle/csvs/Produtos.csv
Centros_Distribuicao             6 linhas  →  /Users/mubotti/Downloads/Portifólio/TechStyle/csvs/Centros_Distribuicao.csv
Estoque                      25200 linhas  →  /Users/mubotti/Downloads/Portifo